In [1]:
import time
from google.cloud import storage
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError
%load_ext google.cloud.bigquery

**Inserts**: Added 2 records to the file:<br>
`{101,"Nachos", 346, 102, 45.6, 0.5, 4, 5, 0.6, 10, 0.6, 0.8, 0.1, 0.9, 0.4, 2, "amc", 2024-03-29 03:53:10.283111 UTC}` <br>
`{102,"Gummy bears", ....}` <br>

**Updates**: Updated calories_from_fat values for 3 existing records.<br>

**Deletes**: removed the two records starting with the substring `Desert`:<br>
`{27,"Add: Sliced Cheese",...}` <br>
`{28,"Angus Burger", ...}` <br>


columns btwn_calories from and fat data_source:
id, name, calories, calories_from_fat, total_fat, saturated_fat, trans_fat, polyunsat_fat, monounsat_fat, cholesterol, sodium, potassium, carbohydrates, dietary_fiber, sugars, protein

In [13]:
%%bigquery
create or replace table movie_raw.amc_snack_copy as
  select * from movie_raw.amc_snack

Query is running:   0%|          |

""


In [ ]:
%%bigquery
create schema if not exists movie_ldg
  options(location="us")

In [10]:
from google.cloud import storage
from google.cloud import bigquery

project_id = "auspicious-lead-382321"
bucket_name = "movie_data_lpez"
folder_name = "incrementals"
dataset_name = "movie_ldg"
region = "us"
file_name = 'amc_snack_032924.csv'
table_name = 'amc_snack_032924'

storage_client = storage.Client()
bq_client = bigquery.Client()

schema = [
  bigquery.SchemaField("id", "INTEGER", mode="REQUIRED"),
  bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("calories", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("calories_from_fat", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("total_fat", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("saturated_fat", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("trans_fat", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("polyunsat_fat", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("monounsat_fat", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("cholesterol", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("sodium", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("potassium", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("carbohydrates", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("dietary_fiber", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("sugars", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("protein", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("data_source", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
]

uri = "gs://{}/{}/{}".format(bucket_name, folder_name, file_name)
table_id = "{}.{}.{}".format(project_id, dataset_name, table_name)

table = bigquery.Table(table_id, schema=schema)
table = bq_client.create_table(table, exists_ok=True)
print("Created table {}".format(table.table_id))

# remove the load_time field from the schema before loading the data,
# the load_time value will be auto-generated
del schema[-1]

job_config = bigquery.LoadJobConfig(
      schema=schema,
      skip_leading_rows=1,
      source_format=bigquery.SourceFormat.CSV,
      write_disposition="WRITE_TRUNCATE",
      field_delimiter=","
    )

load_job = bq_client.load_table_from_uri(uri, table_id, job_config=job_config)
load_job.result()

destination_table = bq_client.get_table(table_id)
print("Loaded {} rows.".format(destination_table.num_rows))

Created table amc_snack_032924
Loaded 100 rows.


## Detect deltas and refresh raw table

##### Identify deltas

In [12]:
%%bigquery
select t.id, t.name, t.calories, t.calories_from_fat, t.total_fat, t.saturated_fat, t.trans_fat, t.polyunsat_fat, t.monounsat_fat, t.cholesterol, t.sodium, t.potassium, t.carbohydrates, t.dietary_fiber, t.sugars, t.protein, t.data_source, r.name as old_description
from movie_ldg.amc_snack_032924 t full join movie_raw.amc_snack_copy r on t.id = r.id
where t.calories != r.calories or t.calories_from_fat != r.calories_from_fat or t.total_fat != r.total_fat and t.saturated_fat != r.saturated_fat or t.trans_fat != r.trans_fat or t.polyunsat_fat != r.polyunsat_fat or t.monounsat_fat != r.monounsat_fat or t.cholesterol != r.cholesterol or t.sodium != r.sodium or t.potassium != r.potassium or t.carbohydrates != r.carbohydrates or t.dietary_fiber != r.dietary_fiber or t.sugars != r.sugars or t.protein != r.protein
or  t.calories_from_fat != r.calories_from_fat or ((t.calories_from_fat IS NULL) = (r.calories_from_fat IS NOT NULL))
order by t.id

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,data_source,old_description
0,<NA>,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,Add: Sliced Cheese
1,<NA>,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,Angus Burger
2,53,Gold Peak Brewed Tea - 16 fl oz,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,15.0,180.0,1.0,NaN,NaN,NaN,amc,Gold Peak Brewed Tea - 16 fl oz
3,54,Gold Peak Brewed Tea - 20 fl oz,5.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,20.0,220.0,2.0,NaN,NaN,NaN,amc,Gold Peak Brewed Tea - 20 fl oz
4,60,BBQ Dipping Sauce,60.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,740.0,NaN,16.0,NaN,5.0,NaN,amc,BBQ Dipping Sauce
5,100,Nachos,346.0,102.0,45.6,0.5,4.0,5.0,0.6,10.0,0.6,0.8,0.1,0.9,0.4,2.0,amc,None
6,101,Gummy Bears,200.0,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,amc,None


##### Process new records (inserts)

In [14]:
%%bigquery
select (select count(*) from movie_ldg.amc_snack_032924) as ldg_count,
(select count(*) from movie_raw.amc_snack) as raw_count

Query is running:   0%|          |

Downloading:   0%|          |

,ldg_count,raw_count
0,100,100


In [15]:
%%bigquery
insert into movie_raw.amc_snack(id, name, calories, calories_from_fat, total_fat, saturated_fat, trans_fat, polyunsat_fat, monounsat_fat, cholesterol, sodium, potassium, carbohydrates, dietary_fiber, sugars, protein, data_source, load_time)
  (select *
  from movie_ldg.amc_snack_032924
  where id not in (select id from movie_raw.amc_snack))

Query is running:   0%|          |

""


In [16]:
%%bigquery
select (select count(*) from movie_ldg.amc_snack_032924) as ldg_count,
(select count(*) from movie_raw.amc_snack) as raw_count

Query is running:   0%|          |

Downloading:   0%|          |

,ldg_count,raw_count
0,100,102


##### Process the changed records (updates)

select t.id, t.calories_from_fat as new_calories_from_fat, t.calories, t.total_fat, t.saturated_fat, t.trans_fat, t.polyunsat_fat, t.monounsat_fat, t.cholesterol, t.sodium, t.potassium, t.carbohydrates, t.dietary_fiber, t.sugars, t.protein, t.data_source, r.name as old_description


In [17]:
%%bigquery
select t.id, t.calories_from_fat as new_calories_from_fat, r.calories_from_fat as old_calories_from_fat
from movie_ldg.amc_snack_032924 t join movie_raw.amc_snack r on t.id = r.id
where t.calories_from_fat != r.calories_from_fat or t.calories_from_fat=0 and r.calories_from_fat IS NULL
order by t.id

Query is running:   0%|          |

Downloading:   0%|          |

,id,new_calories_from_fat,old_calories_from_fat
0,53,0.0,NaN
1,54,0.0,NaN


In [18]:
%%bigquery
update movie_raw.amc_snack r
    set r.calories_from_fat = (select l.calories_from_fat from movie_ldg.amc_snack_032924 l where l.id = r.id),
        r.load_time = (select l.load_time from movie_ldg.amc_snack_032924 l where l.id = r.id)
    where (r.calories_from_fat != (select calories_from_fat from movie_ldg.amc_snack_032924 l where l.id = r.id)
            and r.id = (select id from movie_ldg.amc_snack_032924 l where l.id = r.id))
        or  (r.calories_from_fat IS NULL and (select calories_from_fat from movie_ldg.amc_snack_032924 l where l.id = r.id) = 0
            and r.id = (select id from movie_ldg.amc_snack_032924 l where l.id = r.id))

Query is running:   0%|          |

""


##### Process the deleted records (deletes)

In [19]:
%%bigquery
select r.id as old_id, r.calories_from_fat as old_calories_from_fat, l.id as new_id, l.calories_from_fat as new_calories_from_fat
from movie_ldg.amc_snack_032924 l right join movie_raw.amc_snack r on l.id = r.id
where l.id is null
order by r.id

Query is running:   0%|          |

Downloading:   0%|          |

,old_id,old_calories_from_fat,new_id,new_calories_from_fat
0,27,50.0,<NA>,NaN
1,28,230.0,<NA>,NaN


In [20]:
%%bigquery
delete from movie_raw.amc_snack r
where r.id not in (select l.id from movie_ldg.amc_snack_032924 l)

Query is running:   0%|          |

""


In [19]:
%%bigquery
select r.id as old_id, r.calories_from_fat as old_calories_from_fat, l.id as new_id, l.calories_from_fat as new_calories_from_fat
from movie_ldg.amc_snack_032924 l right join movie_raw.amc_snack r on l.id = r.id
where l.id is null
order by r.id

Query is running:   0%|          |

Downloading: |          |

,old_id,old_calories_from_fat,new_id,new_calories_from_fat


In [20]:
%%bigquery
select * from movie_raw.amc_snack
order by load_time desc

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,data_source,load_time
0,53,Gold Peak Brewed Tea - 16 fl oz,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,15.0,180.0,1.0,NaN,NaN,NaN,amc,2024-03-30 22:16:36.240450+00:00
1,54,Gold Peak Brewed Tea - 20 fl oz,5.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,20.0,220.0,2.0,NaN,NaN,NaN,amc,2024-03-30 22:16:36.240450+00:00
2,100,Nachos,346.0,102.0,45.6,0.5,4.0,5.0,0.6,10.0,0.6,0.8,0.1,0.9,0.4,2.0,amc,2024-03-29 20:47:08.999627+00:00
3,101,Gummy Bears,200.0,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,amc,2024-03-29 20:47:08.999627+00:00
4,71,Pretzel Bites: Add Cinnamon Sugar,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,1.0,NaN,NaN,NaN,amc,2024-02-07 03:53:10.283111+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,8,Plain Popcorn Refillable Tub(No Butter)190 fl ...,1090.0,440.0,49.0,4.0,0.0,15.0,28.0,NaN,2650.0,630.0,148.0,28.0,2.0,25.0,amc,2024-02-07 03:53:10.283111+00:00
96,39,Chili (2.5 Oz),110.0,60.0,7.0,2.0,0.0,NaN,NaN,20.0,620.0,NaN,6.0,1.0,1.0,6.0,amc,2024-02-07 03:53:10.283111+00:00
97,66,Tostitos Cheese Sauce,160.0,100.0,11.0,1.0,NaN,NaN,NaN,5.0,620.0,NaN,15.0,1.0,1.0,1.0,amc,2024-02-07 03:53:10.283111+00:00
98,26,$1 Dog,270.0,130.0,15.0,5.0,NaN,NaN,NaN,30.0,730.0,NaN,26.0,1.0,4.0,9.0,amc,2024-02-07 03:53:10.283111+00:00


## Create copy of staging table
#### Don't re-run this cell after mutating the staging table.

In [21]:
%%bigquery
create or replace table movie_stg.Snack_copy as
  select * from movie_stg.Snack

Query is running:   0%|          |

""


## Re-create the staging table
Note: This logic is the same as before (and was copied from previous notebooks).

In [43]:
%%bigquery
create or replace table movie_stg.Snack as
    select distinct * except(load_time), load_time
    from movie_raw.amc_snack

Query is running:   0%|          |

""


In [14]:
%%bigquery
select (select count(*) from movie_raw.amc_snack) as raw_count,
(select count(*) from movie_stg.Snack) as staging_count

Query is running:   0%|          |

Downloading:   0%|          |

,raw_count,staging_count
0,100,100


In [44]:
%%bigquery
alter table movie_stg.Snack
  add primary key (id) not enforced;

Query is running:   0%|          |

""


In [26]:
%%bigquery
select id, count(*) duplicate_records
from movie_stg.Snack
group by id
having count(*) > 1
order by count(*) desc

Query is running:   0%|          |

Downloading: |          |

,id,duplicate_records


## Create copy of target table (step 13)
#### Don't re-run this cell after mutating the target table.

## Merge staging into target table (step 14)

In [21]:
%%bigquery
create or replace table movie_csp.Snack as
  select * from movie_csp.Snack_copy

Query is running:   0%|          |

""


In [22]:
%%bigquery
select count(*) as num_records from movie_csp.Snack

Query is running:   0%|          |

Downloading:   0%|          |

,num_records
0,100


In [23]:
%%bigquery
select distinct effective_time, discontinue_time, status_flag
from movie_csp.Snack

Query is running:   0%|          |

Downloading:   0%|          |

,effective_time,discontinue_time,status_flag
0,2024-02-20 05:08:39.286737+00:00,NaT,True


##### Merge in the new records and the deleted records:

In [24]:
%%bigquery
merge movie_csp.Snack t
using movie_stg.Snack s
on t.id = s.id
-- handle deleted records
when not matched by source then
  update set discontinue_time = current_timestamp(), status_flag = false
-- handle new records
when not matched by target then
  insert (id, name, calories, calories_from_fat, total_fat, saturated_fat, trans_fat, polyunsat_fat, monounsat_fat, cholesterol, sodium, potassium, carbohydrates, dietary_fiber, sugars, protein, data_source, load_time, effective_time, status_flag)
  values (s.id, s.name, s.calories, s.calories_from_fat, s.total_fat, s.saturated_fat, s.trans_fat, s.polyunsat_fat, s.monounsat_fat, s.cholesterol, s.sodium, potassium, s.carbohydrates, s.dietary_fiber, sugars, s.protein, s.data_source, s.load_time, current_timestamp(), true)

Query is running:   0%|          |

""


In [25]:
%%bigquery
select distinct effective_time, discontinue_time, status_flag
from movie_csp.Snack

Query is running:   0%|          |

Downloading:   0%|          |

,effective_time,discontinue_time,status_flag
0,2024-02-20 05:08:39.286737+00:00,NaT,True
1,2024-04-11 00:13:28.408549+00:00,NaT,True
2,2024-02-20 05:08:39.286737+00:00,2024-04-11 00:13:28.408549+00:00,False


##### Now handle the updated records:

In [26]:
%%bigquery
select s.*
from movie_csp.Snack t join movie_stg.Snack s
on t.id = s.id
where s.name != t.name
or s.calories_from_fat != t.calories_from_fat
or ((s.calories_from_fat IS NULL) = (t.calories_from_fat IS NOT NULL))

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,data_source,load_time,category,is_healthy
0,54,Gold Peak Brewed Tea - 20 fl oz,5.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,20.0,220.0,2.0,NaN,NaN,NaN,amc_ai,2024-03-30 22:16:36.240450+00:00,Beverages,true
1,53,Gold Peak Brewed Tea - 16 fl oz,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,15.0,180.0,1.0,NaN,NaN,NaN,amc_ai,2024-03-30 22:16:36.240450+00:00,Beverages,true


In [27]:
%%bigquery
declare current_ts TIMESTAMP;
set current_ts = current_timestamp();

create temp table updates as
  select s.*
  from movie_csp.Snack t join movie_stg.Snack s
  on t.id = s.id
  where s.name != t.name
or s.calories_from_fat != t.calories_from_fat
or ((s.calories_from_fat IS NULL) = (t.calories_from_fat IS NOT NULL));


update movie_csp.Snack
set discontinue_time = timestamp_sub(current_ts, interval 1 second), status_flag = false
where id in (select id from updates);

insert into movie_csp.Snack
  (id, name, calories, calories_from_fat, total_fat, saturated_fat, trans_fat, polyunsat_fat, monounsat_fat, cholesterol, sodium, potassium, carbohydrates, dietary_fiber, sugars, protein, data_source, load_time, effective_time, status_flag)
    (select id, name, calories, calories_from_fat, total_fat, saturated_fat, trans_fat, polyunsat_fat, monounsat_fat, cholesterol, sodium, potassium, carbohydrates, dietary_fiber, sugars, protein,data_source, load_time, current_ts, true
    from updates);

Query is running:   0%|          |

""


In [28]:
%%bigquery
select * from movie_csp.Snack
where id = 53

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,...,potassium,carbohydrates,dietary_fiber,sugars,protein,data_source,load_time,effective_time,discontinue_time,status_flag
0,53,Gold Peak Brewed Tea - 16 fl oz,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,...,180.0,1.0,NaN,NaN,NaN,amc_ai,2024-03-30 22:16:36.240450+00:00,2024-04-11 00:13:36.582956+00:00,NaT,True
1,53,Gold Peak Brewed Tea - 16 fl oz,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,180.0,1.0,NaN,NaN,NaN,amc,2024-02-07 03:53:10.283111+00:00,2024-02-20 05:08:39.286737+00:00,2024-04-11 00:13:35.582956+00:00,False


Executing query with job ID: 487a5aef-903b-42f2-a91d-d7e0d19880ea
Query executing: 0.75s


ERROR:
 404 Not found: Dataset auspicious-lead-382321:movie_l was not found in location US; reason: notFound, message: Not found: Dataset auspicious-lead-382321:movie_l was not found in location US

Location: US
Job ID: 487a5aef-903b-42f2-a91d-d7e0d19880ea

